In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Environment is working!")

In [ ]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df["Attrition"].value_counts()

In [ ]:
df.describe()

In [ ]:
df.columns.tolist()

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
categorical_cols

In [ ]:
numerical_cols = df.select_dtypes(include="number").columns.tolist()
numerical_cols

In [ ]:
for col in ["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"]:
    print(f"\n{col}:")
    print(df[col].nunique())
    print(df[col].unique()[:10])

In [ ]:
drop_cols = [
    "EmployeeCount",
    "StandardHours",
    "Over18",
    "EmployeeNumber"
]

drop_cols

In [ ]:
pd.crosstab(df["OverTime"], df["Attrition"])

In [ ]:
pd.crosstab(df["OverTime"], df["Attrition"], normalize="index") * 100

In [ ]:
sns.countplot(data=df, x="OverTime", hue="Attrition")

plt.title("Attrition by Overtime")
plt.xlabel("Overtime")
plt.ylabel("Number of Employees")
plt.show()

In [ ]:
attrition_rate_overtime = (
    pd.crosstab(
        df["OverTime"],
        df["Attrition"],
        normalize="index"
    )["Yes"] * 100
)

attrition_rate_overtime

In [ ]:
plt.figure(figsize=(7, 5))

sns.barplot(
    x=attrition_rate_overtime.index,
    y=attrition_rate_overtime.values
)

plt.title("Attrition Rate by Overtime")
plt.xlabel("Overtime")
plt.ylabel("Attrition Rate (%)")
plt.ylim(0, 35)

plt.show()

In [ ]:
jobrole_attrition = (
    pd.crosstab(
        df["JobRole"],
        df["Attrition"],
        normalize="index"
    )["Yes"] * 100
)

jobrole_attrition.sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))

jobrole_attrition.sort_values().plot(kind="barh")

plt.title("Attrition Rate by Job Role")
plt.xlabel("Attrition Rate (%)")
plt.ylabel("Job Role")
plt.xlim(0, 45)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="Attrition",
    y="Age"
)

plt.title("Age Distribution by Attrition")
plt.xlabel("Attrition")
plt.ylabel("Age")

plt.show()

In [ ]:
df.groupby("Attrition")["Age"].agg(["mean", "median", "std"])

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="Attrition",
    y="MonthlyIncome"
)

plt.title("Monthly Income Distribution by Attrition")
plt.xlabel("Attrition")
plt.ylabel("Monthly Income")

plt.show()

In [ ]:
df.groupby("Attrition")["MonthlyIncome"].agg(["mean", "median", "std"])

In [ ]:
satisfaction_attrition = (
    pd.crosstab(
        df["JobSatisfaction"],
        df["Attrition"],
        normalize="index"
    )["Yes"] * 100
)

satisfaction_attrition

In [ ]:
business_travel_attrition = (
    pd.crosstab(
        df["BusinessTravel"],
        df["Attrition"],
        normalize="index"
    )["Yes"] * 100
)

business_travel_attrition.sort_values(ascending=False)

In [ ]:
correlation_matrix = df[numerical_cols].corr()

correlation_matrix["Age"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(16, 12))

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap of Numerical Features")
plt.show()

In [ ]:
# Columns that should not be used as model features
drop_cols = [
    "EmployeeCount",
    "StandardHours",
    "Over18",
    "EmployeeNumber"
]

# Categorical features
categorical_cols = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "OverTime"
]

# Continuous numerical features
numerical_cols = [
    "Age",
    "DailyRate",
    "DistanceFromHome",
    "HourlyRate",
    "MonthlyIncome",
    "MonthlyRate",
    "NumCompaniesWorked",
    "PercentSalaryHike",
    "TotalWorkingYears",
    "TrainingTimesLastYear",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager"
]

# Ordinal features
ordinal_cols = [
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "WorkLifeBalance"
]

target_col = "Attrition"

In [ ]:
feature_cols = categorical_cols + numerical_cols + ordinal_cols

print("Number of features:", len(feature_cols))
print("Number of excluded columns:", len(drop_cols))
print("Total accounted columns:", len(feature_cols) + len(drop_cols) + 1)

In [ ]:
X = df[feature_cols].copy()
y = df[target_col].copy()

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
print(X.head())
print()
print(y.head())

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

numerical_transformer = StandardScaler()

ordinal_transformer = StandardScaler()

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_cols),
        ("numerical", numerical_transformer, numerical_cols),
        ("ordinal", ordinal_transformer, ordinal_cols)
    ]
)

In [ ]:
preprocessor

In [ ]:
preprocessor.fit(X_train)

In [ ]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print("X_train original:", X_train.shape)
print("X_train processed:", X_train_processed.shape)

print("X_test original:", X_test.shape)
print("X_test processed:", X_test_processed.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression(
    max_iter=1000
)

In [ ]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
print(y_pred[:20])

In [ ]:
print(pd.Series(y_pred).value_counts())

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
from sklearn.metrics import roc_auc_score

y_proba = pipeline.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)

print("ROC-AUC:", roc_auc)

In [ ]:
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

In [ ]:
balanced_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", balanced_model)
    ]
)

In [ ]:
balanced_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_balanced = balanced_pipeline.predict(X_test)

In [ ]:
print(pd.Series(y_pred_balanced).value_counts())

In [ ]:
print(classification_report(y_test, y_pred_balanced))

In [ ]:
y_proba_balanced = balanced_pipeline.predict_proba(X_test)[:, 1]

roc_auc_balanced = roc_auc_score(
    y_test,
    y_proba_balanced
)

print("ROC-AUC:", roc_auc_balanced)

In [ ]:
cm_balanced = confusion_matrix(y_test, y_pred_balanced)

print(cm_balanced)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    random_state=42
)

In [ ]:
tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_model)
    ]
)

In [ ]:
tree_pipeline.fit(X_train, y_train)

In [70]:
y_pred_tree = tree_pipeline.predict(X_test)

In [71]:
print(classification_report(y_test, y_pred_tree))

              precision    recall  f1-score   support

          No       0.88      0.86      0.87       247
         Yes       0.33      0.36      0.34        47

    accuracy                           0.78       294
   macro avg       0.60      0.61      0.61       294
weighted avg       0.79      0.78      0.78       294



In [72]:
confusion_matrix(y_test, y_pred_tree)

array([[212,  35],
       [ 30,  17]])

In [73]:
y_proba_tree = tree_pipeline.predict_proba(X_test)[:, 1]

roc_auc_tree = roc_auc_score(
    y_test,
    y_proba_tree
)

print("ROC-AUC:", roc_auc_tree)

ROC-AUC: 0.6100008614006374


In [74]:
print("Tree depth:", tree_pipeline.named_steps["model"].get_depth())
print("Number of leaves:", tree_pipeline.named_steps["model"].get_n_leaves())

Tree depth: 18
Number of leaves: 139


In [75]:
tree_pipeline.named_steps["model"].get_depth()

18

In [76]:
y_pred_tree_train = tree_pipeline.predict(X_train)

print(classification_report(y_train, y_pred_tree_train))

              precision    recall  f1-score   support

          No       1.00      1.00      1.00       986
         Yes       1.00      1.00      1.00       190

    accuracy                           1.00      1176
   macro avg       1.00      1.00      1.00      1176
weighted avg       1.00      1.00      1.00      1176



In [77]:
tree_depth_3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

In [78]:
tree_depth_3_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_depth_3)
    ]
)
tree_depth_3_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['No','Yes']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](30,)","['BusinessTravel','Department','EducationField',..., 'RelationshipSatisfaction','StockOptionLevel','WorkLifeBalance']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped

In [79]:
tree_depth_3_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['No','Yes']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](30,)","['BusinessTravel','Department','EducationField',..., 'RelationshipSatisfaction','StockOptionLevel','WorkLifeBalance']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped

In [80]:
y_pred_depth_3_train = tree_depth_3_pipeline.predict(X_train)

print(classification_report(y_train, y_pred_depth_3_train))

              precision    recall  f1-score   support

          No       0.88      0.97      0.92       986
         Yes       0.70      0.31      0.43       190

    accuracy                           0.87      1176
   macro avg       0.79      0.64      0.68      1176
weighted avg       0.85      0.87      0.85      1176



In [81]:
y_pred_depth_3_test = tree_depth_3_pipeline.predict(X_test)

print(classification_report(y_test, y_pred_depth_3_test))

              precision    recall  f1-score   support

          No       0.86      0.96      0.91       247
         Yes       0.48      0.21      0.29        47

    accuracy                           0.84       294
   macro avg       0.67      0.58      0.60       294
weighted avg       0.80      0.84      0.81       294



In [82]:
y_proba_depth_3 = tree_depth_3_pipeline.predict_proba(X_test)[:, 1]

roc_auc_depth_3 = roc_auc_score(
    y_test,
    y_proba_depth_3
)

print("ROC-AUC:", roc_auc_depth_3)

ROC-AUC: 0.7337841330002584


In [83]:
tree_depth_5 = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

In [84]:
tree_depth_5_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_depth_5)
    ]
)
tree_depth_5_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['No','Yes']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](30,)","['BusinessTravel','Department','EducationField',..., 'RelationshipSatisfaction','StockOptionLevel','WorkLifeBalance']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped

In [85]:
y_pred_depth_5_train = tree_depth_5_pipeline.predict(X_train)

print(classification_report(y_train, y_pred_depth_5_train))

              precision    recall  f1-score   support

          No       0.89      0.99      0.94       986
         Yes       0.91      0.36      0.52       190

    accuracy                           0.89      1176
   macro avg       0.90      0.68      0.73      1176
weighted avg       0.89      0.89      0.87      1176



In [86]:
y_pred_depth_5_test = tree_depth_5_pipeline.predict(X_test)

print(classification_report(y_test, y_pred_depth_5_test))

              precision    recall  f1-score   support

          No       0.86      0.96      0.91       247
         Yes       0.47      0.19      0.27        47

    accuracy                           0.84       294
   macro avg       0.67      0.58      0.59       294
weighted avg       0.80      0.84      0.81       294



In [87]:
y_proba_depth_5 = tree_depth_5_pipeline.predict_proba(X_test)[:, 1]

roc_auc_depth_5 = roc_auc_score(
    y_test,
    y_proba_depth_5
)

print("ROC-AUC:", roc_auc_depth_5)

ROC-AUC: 0.6591437677663883


In [88]:
tree_depth_7 = DecisionTreeClassifier(
    max_depth=7,
    random_state=42
)
tree_depth_7_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_depth_7)
    ]
)
tree_depth_7_pipeline.fit(X_train, y_train)
y_pred_depth_7_train = tree_depth_7_pipeline.predict(X_train)

print(classification_report(y_train, y_pred_depth_7_train))

              precision    recall  f1-score   support

          No       0.93      1.00      0.96       986
         Yes       0.98      0.63      0.77       190

    accuracy                           0.94      1176
   macro avg       0.95      0.81      0.87      1176
weighted avg       0.94      0.94      0.93      1176



In [89]:
y_pred_depth_7_test = tree_depth_7_pipeline.predict(X_test)

print(classification_report(y_test, y_pred_depth_7_test))

              precision    recall  f1-score   support

          No       0.88      0.93      0.90       247
         Yes       0.45      0.32      0.38        47

    accuracy                           0.83       294
   macro avg       0.67      0.62      0.64       294
weighted avg       0.81      0.83      0.82       294



In [90]:
y_proba_depth_7 = tree_depth_7_pipeline.predict_proba(X_test)[:, 1]

roc_auc_depth_7 = roc_auc_score(
    y_test,
    y_proba_depth_7
)

print("ROC-AUC:", roc_auc_depth_7)

ROC-AUC: 0.6825307950727881


In [91]:
tree_depth_10 = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)
tree_depth_10_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_depth_10)
    ]
)
tree_depth_10_pipeline.fit(X_train, y_train)
y_pred_depth_10_train = tree_depth_10_pipeline.predict(X_train)

print(classification_report(y_train, y_pred_depth_10_train))

              precision    recall  f1-score   support

          No       0.97      1.00      0.99       986
         Yes       0.99      0.86      0.92       190

    accuracy                           0.98      1176
   macro avg       0.98      0.93      0.96      1176
weighted avg       0.98      0.98      0.98      1176



In [92]:
y_pred_depth_10_test = tree_depth_10_pipeline.predict(X_test)

print(classification_report(y_test, y_pred_depth_10_test))

              precision    recall  f1-score   support

          No       0.88      0.90      0.89       247
         Yes       0.41      0.36      0.39        47

    accuracy                           0.82       294
   macro avg       0.65      0.63      0.64       294
weighted avg       0.81      0.82      0.81       294



In [93]:
y_proba_depth_10 = tree_depth_10_pipeline.predict_proba(X_test)[:, 1]

roc_auc_depth_10 = roc_auc_score(
    y_test,
    y_proba_depth_10
)

print("ROC-AUC:", roc_auc_depth_10)

ROC-AUC: 0.5470324748040313


In [94]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# 1. Create Decision Tree
tree_model_split2 = DecisionTreeClassifier(
    min_samples_split=2,
    random_state=42
)

# 2. Create pipeline using our EXISTING preprocessor
tree_pipeline_split2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_model_split2)
    ]
)

# 3. Train the model
tree_pipeline_split2.fit(X_train, y_train)

# 4. Predictions
y_pred_split2 = tree_pipeline_split2.predict(X_test)

# 5. Probability predictions for ROC-AUC
y_proba_split2 = tree_pipeline_split2.predict_proba(X_test)[:, 1]

# 6. Evaluation
print("Test Accuracy:", accuracy_score(y_test, y_pred_split2))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_split2))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_split2))

print(
    "\nROC-AUC:",
    roc_auc_score(y_test, y_proba_split2)
)

Test Accuracy: 0.7789115646258503

Classification Report:
              precision    recall  f1-score   support

          No       0.88      0.86      0.87       247
         Yes       0.33      0.36      0.34        47

    accuracy                           0.78       294
   macro avg       0.60      0.61      0.61       294
weighted avg       0.79      0.78      0.78       294

Confusion Matrix:
[[212  35]
 [ 30  17]]

ROC-AUC: 0.6100008614006374


In [95]:
tree_model_split2 = DecisionTreeClassifier(
    min_samples_split=10,
    random_state=42
)
tree_pipeline_split2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_model_split2)
    ]
)

# 3. Train the model
tree_pipeline_split2.fit(X_train, y_train)

# 4. Predictions
y_pred_split2 = tree_pipeline_split2.predict(X_test)

# 5. Probability predictions for ROC-AUC
y_proba_split2 = tree_pipeline_split2.predict_proba(X_test)[:, 1]

# 6. Evaluation
print("Test Accuracy:", accuracy_score(y_test, y_pred_split2))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_split2))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_split2))

print(
    "\nROC-AUC:",
    roc_auc_score(y_test, y_proba_split2)
)

Test Accuracy: 0.7925170068027211

Classification Report:
              precision    recall  f1-score   support

          No       0.87      0.88      0.88       247
         Yes       0.34      0.32      0.33        47

    accuracy                           0.79       294
   macro avg       0.61      0.60      0.60       294
weighted avg       0.79      0.79      0.79       294

Confusion Matrix:
[[218  29]
 [ 32  15]]

ROC-AUC: 0.6458781979498665


In [96]:
tree_model_split2 = DecisionTreeClassifier(
    min_samples_split=20,
    random_state=42
)
tree_pipeline_split2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_model_split2)
    ]
)

# 3. Train the model
tree_pipeline_split2.fit(X_train, y_train)

# 4. Predictions
y_pred_split2 = tree_pipeline_split2.predict(X_test)

# 5. Probability predictions for ROC-AUC
y_proba_split2 = tree_pipeline_split2.predict_proba(X_test)[:, 1]

# 6. Evaluation
print("Test Accuracy:", accuracy_score(y_test, y_pred_split2))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_split2))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_split2))

print(
    "\nROC-AUC:",
    roc_auc_score(y_test, y_proba_split2)
)

Test Accuracy: 0.8197278911564626

Classification Report:
              precision    recall  f1-score   support

          No       0.86      0.94      0.90       247
         Yes       0.38      0.21      0.27        47

    accuracy                           0.82       294
   macro avg       0.62      0.57      0.59       294
weighted avg       0.79      0.82      0.80       294

Confusion Matrix:
[[231  16]
 [ 37  10]]

ROC-AUC: 0.6547075544835903


In [97]:
tree_model_split2 = DecisionTreeClassifier(
    min_samples_leaf=1,
    random_state=42
)
tree_pipeline_split2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_model_split2)
    ]
)

# 3. Train the model
tree_pipeline_split2.fit(X_train, y_train)

# 4. Predictions
y_pred_split2 = tree_pipeline_split2.predict(X_test)

# 5. Probability predictions for ROC-AUC
y_proba_split2 = tree_pipeline_split2.predict_proba(X_test)[:, 1]

# 6. Evaluation
print("Test Accuracy:", accuracy_score(y_test, y_pred_split2))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_split2))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_split2))

print(
    "\nROC-AUC:",
    roc_auc_score(y_test, y_proba_split2)
)

Test Accuracy: 0.7789115646258503

Classification Report:
              precision    recall  f1-score   support

          No       0.88      0.86      0.87       247
         Yes       0.33      0.36      0.34        47

    accuracy                           0.78       294
   macro avg       0.60      0.61      0.61       294
weighted avg       0.79      0.78      0.78       294

Confusion Matrix:
[[212  35]
 [ 30  17]]

ROC-AUC: 0.6100008614006374


In [98]:
tree_model_split2 = DecisionTreeClassifier(
    min_samples_leaf=10,
    random_state=42
)
tree_pipeline_split2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tree_model_split2)
    ]
)

# 3. Train the model
tree_pipeline_split2.fit(X_train, y_train)

# 4. Predictions
y_pred_split2 = tree_pipeline_split2.predict(X_test)

# 5. Probability predictions for ROC-AUC
y_proba_split2 = tree_pipeline_split2.predict_proba(X_test)[:, 1]

# 6. Evaluation
print("Test Accuracy:", accuracy_score(y_test, y_pred_split2))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_split2))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_split2))

print(
    "\nROC-AUC:",
    roc_auc_score(y_test, y_proba_split2)
)

Test Accuracy: 0.8367346938775511

Classification Report:
              precision    recall  f1-score   support

          No       0.87      0.95      0.91       247
         Yes       0.48      0.23      0.31        47

    accuracy                           0.84       294
   macro avg       0.67      0.59      0.61       294
weighted avg       0.80      0.84      0.81       294

Confusion Matrix:
[[235  12]
 [ 36  11]]

ROC-AUC: 0.6760702902920148


In [99]:
from sklearn.pipeline import Pipeline

dt_cv_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        random_state=42,
        max_depth=3
    ))
])

In [100]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    dt_cv_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
)

print("CV scores:", cv_scores)
print("Mean ROC-AUC:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())

CV scores: [0.62925306 0.67906759 0.71059311 0.6627705  0.74779589]
Mean ROC-AUC: 0.6858960294907396
Standard deviation: 0.0405743308395764


In [101]:
param_grid = {
    "classifier__max_depth": [3, 5, 7],
    "classifier__min_samples_split": [2, 10, 20],
    "classifier__min_samples_leaf": [1, 5, 10]
}

print(param_grid)

{'classifier__max_depth': [3, 5, 7], 'classifier__min_samples_split': [2, 10, 20], 'classifier__min_samples_leaf': [1, 5, 10]}


In [102]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=dt_cv_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__max_depth': [3, 5, ...], 'classifier__min_samples_leaf': [1, 5, ...], 'classifier__min_samples_split': [2, 10, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",T

In [103]:
print("Best Parameters:")
print(grid_search.best_params_)
print("Best CV ROC-AUC:")
print(grid_search.best_score_)

Best Parameters:
{'classifier__max_depth': 5, 'classifier__min_samples_leaf': 10, 'classifier__min_samples_split': 2}
Best CV ROC-AUC:
0.7132070099876672


In [104]:
best_dt = grid_search.best_estimator_

print(best_dt)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['BusinessTravel',
                                                   'Department',
                                                   'EducationField', 'Gender',
                                                   'JobRole', 'MaritalStatus',
                                                   'OverTime']),
                                                 ('numerical', StandardScaler(),
                                                  ['Age', 'DailyRate',
                                                   'DistanceFromHome',
                                                   'HourlyRate',
                                                   'MonthlyIncome',
                                                   'MonthlyRate',
                                 

In [105]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_pred = best_dt.predict(X_test)
y_proba = best_dt.predict_proba(X_test)[:, 1]

print("Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))

Test Accuracy: 0.8435374149659864

Classification Report:
              precision    recall  f1-score   support

          No       0.87      0.96      0.91       247
         Yes       0.53      0.21      0.30        47

    accuracy                           0.84       294
   macro avg       0.70      0.59      0.61       294
weighted avg       0.81      0.84      0.81       294

Confusion Matrix:
[[238   9]
 [ 37  10]]
Test ROC-AUC: 0.6983805668016194


In [110]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", rf_model)
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

roc_auc_rf = roc_auc_score(y_test, y_proba_rf)

print("ROC-AUC:", roc_auc_rf)

Test Accuracy: 0.8435374149659864
              precision    recall  f1-score   support

          No       0.86      0.97      0.91       247
         Yes       0.53      0.17      0.26        47

    accuracy                           0.84       294
   macro avg       0.70      0.57      0.59       294
weighted avg       0.81      0.84      0.81       294

ROC-AUC: 0.8018778533896115
